# Applied Unsupervised Learning Techniques

### Imports

In [ ]:
import altair as alt
import pandas as pd
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

from birds.source_data import nabbp, avonet

### Configuration

In [ ]:
alt.data_transformers.enable("vegafusion")

### Load NABBP Species Data

In [ ]:
def load_nabbp_species() -> pd.DataFrame:
    lookup = nabbp.LookupTables()
    df = lookup.species
    return df[df['ENDANGERED'] == 'Y']
    

nabbp_species_df = load_nabbp_species()
nabbp_species_df.head()

### Load Avonet Bird Tree Data

In [ ]:
def load_avonet_data() -> pd.DataFrame:
    df = avonet.DataTables().bird_tree
    return df

avonet_df = load_avonet_data()
avonet_df.head()

### Merge Avonet and NABBP Datasets

#### Compute dataset overlap using the scientific name column

In [ ]:
def compute_overlap(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame):
    bt= set(avonet_data['Species3'].unique())
    d = set(nabbp_data['SCI_NAME'].unique())

    overlap  = bt.intersection(d)
    return len(overlap)
    

compute_overlap(avonet_df, nabbp_species_df)

### Transform and Clean Merged Dataset

In [ ]:
def merge_datasets(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame) -> pd.DataFrame:
    merged_df = pd.merge(avonet_data, nabbp_data, left_on='Species3', right_on='SCI_NAME', how='left')
    return merged_df


merged_df = merge_datasets(avonet_df, nabbp_species_df)
merged_df.head()

In [ ]:
def transform_merge_dataset(merged_data: pd.DataFrame) -> pd.DataFrame:
    avonet_feature_columns = [
        'Beak.Length_Culmen',
        'Beak.Length_Nares',
        'Beak.Width',
        'Beak.Depth',
        'Tarsus.Length',
        'Wing.Length',
        'Kipps.Distance',
        'Hand-Wing.Index',
        'Tail.Length',
        'Mass',
    ]
    id_columns = [
        'SPECIES_ID',
        'Species3',
        'Family3',
        'Order3',
        'ENDANGERED',
    ]
    columns_to_keep = id_columns + avonet_feature_columns
    cleaned = (
        merged_data[columns_to_keep]
            .rename(columns={
                "ENDANGERED": "is_endangered",
                "SPECIES_ID": "species_id",
                "Species3": "species_name",
                "Family3": "family_name",
                "Order3": "order_name",
                "Beak.Length_Culmen": "beak_length_culmen",
                "Beak.Length_Nares": "beak_length_nares",
                "Beak.Width": "beak_width",
                "Beak.Depth": "beak_depth",
                "Tarsus.Length": "tarsus_length",
                "Wing.Length": "wing_length",
                "Kipps.Distance": "kipps_distance",
                "Hand-Wing.Index": "hand_wing_index",
                "Tail.Length": "tail_length",
                "Mass": "mass",
            })
            .fillna({
                "is_endangered": 'N',
            })
    )
    return cleaned

transformed_df = transform_merge_dataset(merged_df)
transformed_df.head()

In [ ]:
AVONET_FEATURE_COLUMNS = [
    'beak_length_culmen',
    'beak_length_nares',
    'beak_width',
    'beak_depth',
    'tarsus_length',
    'wing_length',
    'kipps_distance',
    'hand_wing_index',
    'tail_length',
    'mass',
]

In [ ]:
# Create a density plot for each numeric feature
charts = []
for col in AVONET_FEATURE_COLUMNS:
    chart = (
        alt.Chart(transformed_df)
            .transform_density(col, as_=[col, 'density'])
            .mark_area(opacity=0.6)
            .encode(x=alt.X(col, title=col), y='density:Q')
            .properties(width=150, height=100)
    )
    charts.append(chart)
alt.concat(*charts, columns=3, title="Density Plots of Avonet Features")

### Feature Scaling

In [ ]:
def scale_feature_columns(data: pd.DataFrame, feature_columns: list[str]):
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[feature_columns])

    data_scaled = data.copy()
    data_scaled[feature_columns] = scaled_features
    
    return data_scaled


scaled_df = scale_feature_columns(transformed_df, AVONET_FEATURE_COLUMNS)
scaled_df.head()

### UMAP Projection for Visualizations

In [ ]:
def compute_umap_projection(data: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    reducer = umap.UMAP(
        random_state=42,
        min_dist=0.1,
        n_neighbors=15,
        metric='euclidean'
    )
    embedding = reducer.fit_transform(data[feature_columns])

    data_umap = data.copy()
    data_umap['umap_component_1'] = embedding[:, 0]
    data_umap['umap_component_2'] = embedding[:, 1]
    return data_umap


In [ ]:
umap_df = compute_umap_projection(scaled_df, AVONET_FEATURE_COLUMNS)
umap_df.head()

### Label Preparation

In [ ]:
# from sklearn.semi_supervised import LabelPropagation

# model = LabelPropagation(kernel="rbf", gamma=0.1, max_iter=5)

# y_semi = labels.copy()
# y_semi[y_semi == 'N'] = -1
# y_semi[y_semi == 'Y'] = 1
# y_semi = y_semi.astype(int)

# model.fit(scaled_data, y_semi)
# y_pred = model.transduction_

In [ ]:
# # Count the occurrences of each label in y_semi
# import numpy as np
# unique, counts = np.unique(y_semi, return_counts=True)
# label_counts = dict(zip(unique, counts))
# print("Label counts in y_semi:", label_counts)

### Visualizations

In [ ]:
def plot_umap(data: pd.DataFrame, color: str, title: str) -> alt.Chart:
    chart = (
    alt.Chart(data)
        .mark_circle(size=30)
        .encode(
            x=alt.X('umap_component_1'),
            y=alt.Y('umap_component_2'),
            color=alt.Color(color, legend=None),
            tooltip=['family_name', 'order_name', 'is_endangered']
        )
        .properties(
            title=title,
            width=600,
            height=500
        )
    )
    return chart

In [ ]:
(plot_umap(umap_df, 'order_name', 'Bird Orders') | plot_umap(umap_df, 'family_name', 'Bird Families')).interactive().resolve_scale(x='shared', y='shared') 

In [ ]:
plot_umap(umap_df, 'is_endangered', 'Endangered Status').interactive()